# VadCLIP Baseline — Runner

Chạy `VadCLIP/baseline/` (code gốc upstream, tách khỏi `VadCLIP/src/`) và gom mọi kết quả về
một bảng duy nhất.

## Chạy theo đúng thứ tự này

| § | Cell | Việc | Thời gian |
|---|---|---|---|
| 1 | 2 | Mount Drive + cấu hình | vài giây |
| 2 | 4 | Cài dependencies | ~1 phút |
| 3 | 6 | **Copy feature sang đĩa local** | vài phút |
| 4 | 8 | **Preflight — kiểm tra mọi thứ trước khi tốn giờ** | vài giây |
| 5 | 10 | Chấm các model đã có (paper, baseline_ctrl, v0) | vài phút |
| 6 | 12 | Train baseline seed mới | **nhiều giờ** |
| 7 | 14, 15 | Đường cong theo epoch + copy về Drive | vài phút |
| 8 | 17 | Bảng tổng hợp | vài giây |

> **§5 quan trọng hơn §6.** Nó sửa được bảng kết quả trong báo cáo mà chỉ tốn vài phút.
> Nếu chỉ làm được một việc, làm §5.

> **Cần GPU.** `layers.py` upstream hardcode `.to('cuda')`.
> Runtime → Change runtime type → GPU.

> **Nếu §4 báo lỗi thiếu cờ** — Drive đang giữ bản `baseline/` cũ. Upload lại thư mục
> `VadCLIP/baseline/` rồi chạy lại §4. Đừng bỏ qua.

## 1. Mount Drive Và Cấu Hình

Mọi hằng số nằm ở đây. Không cell nào phía sau tự định nghĩa đường dẫn hay tham số.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

# ---------------- Đường dẫn trên Drive ----------------
PROJECT_ROOT     = Path('/content/drive/MyDrive/Finetune VadCLIP')
BASELINE_SRC     = PROJECT_ROOT / 'VadCLIP' / 'baseline' / 'src'
LIST_DIR         = PROJECT_ROOT / 'VadCLIP' / 'list'
OLD_MODEL_DIR    = PROJECT_ROOT / 'VadCLIP' / 'src' / 'model'   # model_baseline_ctrl.pth, model_v0.pth
PAPER_CHECKPOINT = PROJECT_ROOT / 'model_ucf.pth'               # trọng số tác giả công bố
DRIVE_RESULT     = PROJECT_ROOT / 'Result' / 'baseline'
LOG_DIR          = DRIVE_RESULT / 'logs'

DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]

# ---------------- Đường dẫn local (nhanh) ----------------
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
OUT_ROOT           = Path('/content/baseline_out')
FEATURE_ROOT       = None          # §3 gán. Để None nên §4 bắt được nếu bỏ qua §3.

# ---------------- List + ground truth (tính từ baseline/src) ----------------
TRAIN_LIST = '../../list/ucf_CLIP_rgb_relative.csv'
TEST_LIST  = '../../list/ucf_CLIP_rgbtest_relative.csv'
GT_ARGS = [
    '--gt-path',         '../../list/gt_ucf.npy',
    '--gt-segment-path', '../../list/gt_segment_ucf.npy',
    '--gt-label-path',   '../../list/gt_label_ucf.npy',
]

# ================= CẤU HÌNH BASELINE CHUẨN =================
# Dùng y hệt cho mọi lần chạy. Đổi một giá trị ở đây là đổi cho tất cả.
SEEDS           = [777]        # thêm seed nếu chạy được nhiều lần
BASELINE_LR     = '2e-5'       # KHÔNG phải 1e-5 của paper — xem README, để làm thí nghiệm riêng
BASELINE_EPOCHS = 10
BASELINE_BATCH  = 64
BASELINE_FLAGS  = ['--num-workers', '4', '--pin-memory', 'true', '--deterministic', 'true']
# ===========================================================

# Số của paper, dùng làm dòng tham chiếu và để kiểm tra thước đo.
PAPER_METRICS = {'AUC1': 88.02, 'AP1': 33.56, 'AUC2': 85.69, 'AP2': 26.50, 'avgMAP': 6.68}

os.chdir(BASELINE_SRC)
sys.path.insert(0, str(BASELINE_SRC))
LOG_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

PY = [sys.executable, '-u']


def run_command(cmd, log_name=None):
    '''Chạy lệnh, stream từng dòng, ghi log lên Drive ngay khi có dòng mới.

    Ghi tăng dần chứ không gom cuối, nên session rớt giữa chừng vẫn còn log để đọc.
    '''
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    log_file = open(LOG_DIR / log_name, 'w', encoding='utf-8') if log_name else None
    if log_file:
        log_file.write('$ ' + ' '.join(cmd) + chr(10))
        log_file.flush()
    captured = []
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            captured.append(line)
            if log_file:
                log_file.write(line)
                log_file.flush()
    finally:
        process.wait()
        if log_file:
            log_file.close()
    if log_name:
        print('Log:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return ''.join(captured)


def score_model(model_path, name):
    '''Chấm điểm một file trọng số bằng ucf_test.py — thước đo duy nhất của notebook này.'''
    preflight()
    return run_command(PY + [
        'ucf_test.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--model-path', model_path,
    ], log_name=f'score_{name}.log')


def train_baseline(seed):
    '''Train một baseline với cấu hình chuẩn ở trên. Trả về tag.'''
    preflight()
    tag = f'baseline_seed{seed}'
    out = OUT_ROOT / tag
    out.mkdir(parents=True, exist_ok=True)
    run_command(PY + [
        'ucf_train.py',
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list',  TEST_LIST,
        *GT_ARGS,
        '--seed', seed,
        '--lr', BASELINE_LR,
        '--max-epoch', BASELINE_EPOCHS,
        '--batch-size', BASELINE_BATCH,
        # tên riêng, KHÔNG đặt là model_ucf.pth để khỏi lẫn với checkpoint của tác giả
        '--model-path',           out / f'model_{tag}.pth',
        '--checkpoint-path',      out / 'checkpoint.pth',
        '--save-cur-path',        out / 'model_cur.pth',
        '--epoch-checkpoint-dir', out / 'epoch_checkpoints',
    ] + BASELINE_FLAGS, log_name=f'train_{tag}.log')
    return tag


print('Baseline src :', BASELINE_SRC)
print('cwd          :', Path.cwd())
print('Kết quả về   :', DRIVE_RESULT)
print()
print('Cấu hình chuẩn: seeds', SEEDS, '| lr', BASELINE_LR, '| epoch', BASELINE_EPOCHS,
      '| batch', BASELINE_BATCH)
print('               ', ' '.join(BASELINE_FLAGS))
print()
print('FEATURE_ROOT chưa gán — chạy §3 (cell copy feature) trước.')

## 2. Cài Dependencies

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy pandas matplotlib

## 3. Copy Feature Sang Đĩa Local

**Không được bỏ qua.** Đọc thẳng 16.100 file `.npy` từ Drive, mười epoch liền, sẽ chậm gấp nhiều lần.

Cell này gán `FEATURE_ROOT`. Trước khi chạy nó, `FEATURE_ROOT is None` và §4 sẽ chặn lại.

In [ ]:
import shutil
import time

subprocess.run(['df', '-h', '/content'], check=False)
archive = next((p for p in DRIVE_FEATURE_ARCHIVES if p.exists()), None)

start = time.time()
if LOCAL_FEATURE_ROOT.exists() and any(LOCAL_FEATURE_ROOT.iterdir()):
    print('Feature đã có sẵn ở local:', LOCAL_FEATURE_ROOT)
elif archive is not None:
    local_archive = Path('/content') / archive.name
    if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
        print('Copying archive:', archive)
        shutil.copy2(archive, local_archive)
    print('Extracting:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)
    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive phải chứa thư mục top-level UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run(['rsync', '-ah', '--info=progress2',
                        f'{DRIVE_FEATURE_ROOT}/', f'{LOCAL_FEATURE_ROOT}/'], check=True)
    else:
        subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'Done in {time.time() - start:.1f}s')
print('FEATURE_ROOT ->', FEATURE_ROOT)

## 4. Preflight

Kiểm tra sáu thứ trước khi tiêu tốn hàng giờ GPU. `preflight()` được gọi tự động ở đầu mọi
cell train và chấm điểm, nên không thể chạy nhầm với setup hỏng.

Nếu báo **thiếu cờ `--deterministic`** → Drive đang giữ bản `baseline/` cũ. Upload lại thư mục
`VadCLIP/baseline/` (khoảng 1.4 MB) rồi chạy lại cell này. Drive có thể sync chậm vài phút.

In [ ]:
import csv
import numpy as np
import torch
from collections import Counter

_preflight_done = False


def preflight(force=False):
    global _preflight_done
    if _preflight_done and not force:
        return True

    problems = []

    # 1. GPU
    if not torch.cuda.is_available():
        problems.append('Không có GPU. layers.py upstream hardcode .to("cuda"). '
                        'Runtime -> Change runtime type -> GPU.')

    # 2. FEATURE_ROOT phải là đĩa local
    if FEATURE_ROOT is None:
        problems.append('FEATURE_ROOT chưa gán — chạy §3 (copy feature) trước.')
    elif str(FEATURE_ROOT).startswith('/content/drive'):
        problems.append(f'FEATURE_ROOT đang trỏ vào Drive ({FEATURE_ROOT}). Chạy §3 để copy '
                        'sang /content, nếu không train sẽ rất chậm.')

    # 3. File code + list
    need = [BASELINE_SRC / n for n in
            ['ucf_train.py', 'ucf_test.py', 'ucf_option.py', 'model.py',
             'utils/dataset.py', 'utils/tools.py', 'utils/layers.py',
             'utils/ucf_detectionMAP.py', 'clip/clip.py', 'clip/bpe_simple_vocab_16e6.txt.gz']]
    need += [LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv']
    for p in need:
        if not p.exists():
            problems.append(f'Thiếu file: {p}')

    # 4. Ground truth
    for p in [LIST_DIR / n for n in ['gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy']]:
        if not p.exists():
            problems.append(f'Thiếu ground truth: {p}')

    # 5. baseline/ trên Drive phải là bản mới (có đủ ba cờ)
    if (BASELINE_SRC / 'ucf_train.py').exists():
        helptext = subprocess.run(PY + ['ucf_train.py', '--help'],
                                  capture_output=True, text=True).stdout
        for flag in ['--num-workers', '--pin-memory', '--deterministic', '--feature-root']:
            if flag not in helptext:
                problems.append(f'ucf_train.py trên Drive thiếu cờ {flag} — bản cũ. '
                                'Upload lại thư mục VadCLIP/baseline/.')

    # 6. Độ phủ feature
    if FEATURE_ROOT is not None and not str(FEATURE_ROOT).startswith('/content/drive'):
        for name, expected in [('ucf_CLIP_rgb_relative.csv', 16100),
                               ('ucf_CLIP_rgbtest_relative.csv', 290)]:
            path = LIST_DIR / name
            if not path.exists():
                continue
            rows = list(csv.DictReader(open(path, encoding='utf-8')))
            missing = [r for r in rows if not (FEATURE_ROOT / r['path']).exists()]
            print(f'  {name}: {len(rows)} dòng (mong đợi {expected}), thiếu {len(missing)}')
            if missing:
                print('    Thiếu theo nhãn:', dict(Counter(r['label'] for r in missing)))
                for r in missing[:5]:
                    print('      ', r['path'])
                problems.append(f'{name}: thiếu {len(missing)} file feature.')

    if problems:
        print()
        print('PREFLIGHT KHÔNG ĐẠT:')
        for p in problems:
            print('  -', p)
        raise RuntimeError('Sửa các mục trên rồi chạy lại cell này.')

    print()
    print('PREFLIGHT ĐẠT')
    print('  GPU          :', torch.cuda.get_device_name(0))
    print('  torch        :', torch.__version__)
    print('  FEATURE_ROOT :', FEATURE_ROOT)
    gt = np.load(LIST_DIR / 'gt_ucf.npy')
    print('  gt_ucf.npy   :', len(gt), 'frame |', int(gt.sum()), 'frame bất thường')
    _preflight_done = True
    return True


preflight(force=True)

## 5. Chấm Các Model Đã Có

**Rẻ và quan trọng.** Vài phút, và nó sửa được bảng kết quả trong báo cáo.

Ba model, cùng một thước đo `ucf_test.py`:

| Model | Là gì | Mong đợi |
|---|---|---|
| `model_ucf.pth` | trọng số **tác giả công bố** | phải ra đúng **88.02 / 6.68** |
| `model_baseline_ctrl.pth` | bạn train, ràng buộc **TẮT** (λ=0) → baseline | ~88.13 / 8.50 |
| `model_v0.pth` | bạn train, ràng buộc **BẬT** (λ=0.01) → thí nghiệm | chưa biết |

Model đầu tiên là **quả cân chuẩn**. Nếu nó ra đúng 88.02 thì thước đo, ground truth, test list
và bộ prompt đều đúng — và mọi con số còn lại đáng tin.

> Nhớ: `baseline_ctrl` là λ=**0**, `v0` là λ=**0.01**. Đừng đảo hai cái này.

In [ ]:
TARGETS = [
    (PAPER_CHECKPOINT,                        'paper'),
    (OLD_MODEL_DIR / 'model_baseline_ctrl.pth', 'baseline_ctrl'),
    (OLD_MODEL_DIR / 'model_v0.pth',            'v0'),
]

for path, name in TARGETS:
    print('#' * 78)
    print(name, '->', path)
    if not Path(path).exists():
        print('  BỎ QUA: không tìm thấy file.')
        continue
    score_model(path, name)

In [ ]:
# Kiểm tra thước đo: bản chấm điểm có tái lập đúng số của paper không?
import re

def read_score(name):
    path = LOG_DIR / f'score_{name}.log'
    if not path.exists():
        return None
    text = path.read_text(encoding='utf-8')
    def last(p, scale=1.0):
        hits = re.findall(p, text)
        return float(hits[-1]) * scale if hits else float('nan')
    return {'AUC1': last(r'AUC1:\s+([\d.]+)', 100), 'AP1': last(r'AP1:\s+([\d.]+)', 100),
            'AUC2': last(r'AUC2:\s+([\d.]+)', 100), 'AP2': last(r'AP2:\s*([\d.]+)', 100),
            'avgMAP': last(r'average MAP:\s+([\d.]+)')}


paper = read_score('paper')
if paper is None:
    print('Chưa chấm checkpoint tác giả — chạy cell trên trước.')
else:
    print('KIỂM TRA THƯỚC ĐO trên checkpoint tác giả:')
    ok = True
    for key, expected in PAPER_METRICS.items():
        got = paper[key]
        good = abs(got - expected) < 0.05
        ok = ok and good
        print(f'  {"OK  " if good else "LỆCH"} {key:7s} đo được {got:6.2f}   paper {expected:6.2f}')
    print()
    print('=> Thước đo chuẩn. Mọi con số khác trong notebook này đáng tin.' if ok else
          '=> Thước đo LỆCH. Dừng lại: kiểm tra ground truth và test list trước khi đọc số nào khác.')

## 6. Train Baseline Seed Mới

Cấu hình lấy từ §1, không nhận tham số riêng ở đây — để mọi lần chạy giống hệt nhau trừ seed.

```
lr 2e-5 · batch 64 · 10 epoch · MultiStepLR([4,8], 0.1)
--num-workers 4 --pin-memory true --deterministic true
```

Vì sao chọn các giá trị này:

- **lr 2e-5** là mặc định repo. Hai lần chạy trước cho 87.55 và 88.13, ôm trọn 88.02 của paper.
  Paper ghi 1e-5 nhưng chưa ai thử — để làm thí nghiệm riêng, đừng trộn vào baseline.
- **num-workers 4** khớp với `baseline_ctrl` và `v0`, nên mọi thứ so được với nhau.
  Chọn một giá trị rồi **không đổi nữa** — nó làm đổi thứ tự lô dữ liệu, tức đổi cả model.
- **deterministic true** để chạy lại cùng lệnh ra cùng kết quả. Upstream comment mất dòng
  `cudnn.deterministic` nên vốn không tái lập được.

Theo dõi dòng `=== end of epoch N | best AUC so far: ... ===`.
Log ghi lên Drive từng dòng nên rớt session vẫn đọc được.

> Đây là **điểm dữ liệu thứ ba**, không phải nỗ lực đạt tới một con số. Bạn đã có 87.55 và 88.13.
> Ba điểm là tối thiểu để viết `trung bình ± độ lệch chuẩn`.
> Đừng chạy đi chạy lại rồi chọn lần đẹp nhất — đó là chọn kết quả theo tập test.

In [ ]:
trained_tags = []
for seed in SEEDS:
    print('#' * 78)
    print('TRAIN seed', seed)
    tag = train_baseline(seed)
    trained_tags.append(tag)
    score_model(OUT_ROOT / tag / f'model_{tag}.pth', tag)
print()
print('Đã train:', trained_tags)

## 7. Đường Cong Theo Epoch

`ucf_train.py` chấm toàn tập test ~12 lần mỗi epoch, nên log đã chứa sẵn đường cong.

`AUC1` cao nhất phải bằng con số §6 vừa in ra — vì file trọng số cuối chính là checkpoint tốt nhất đó.

> `ucf_train.py` chọn checkpoint theo AUC **trên chính tập test**. Đây là chọn-model-trên-test
> nên con số lạc quan hơn thực tế. Giữ nguyên vì đó là hành vi upstream, nhưng phải ghi rõ khi báo cáo.

In [ ]:
import pandas as pd

TRAIN_CURVE_RE = re.compile(
    r'epoch:\s+(\d+)\s+\|\s+step:\s+(\d+).*?'
    r'AUC1:\s+([\d.]+)\s+AP1:\s+([\d.]+).*?'
    r'AUC2:\s+([\d.]+)\s+AP2:\s+([\d.]+).*?'
    r'average MAP:\s+([\d.]+)', re.S)


def training_curve(tag):
    text = (LOG_DIR / f'train_{tag}.log').read_text(encoding='utf-8')
    return pd.DataFrame([
        dict(epoch=int(a), step=int(b), AUC1=float(c) * 100, AP1=float(d) * 100,
             AUC2=float(e) * 100, AP2=float(f) * 100, avgMAP=float(g))
        for a, b, c, d, e, f, g in TRAIN_CURVE_RE.findall(text)])


for tag in (trained_tags or [f'baseline_seed{s}' for s in SEEDS]):
    if not (LOG_DIR / f'train_{tag}.log').exists():
        print('Chưa có log cho', tag)
        continue
    curve = training_curve(tag)
    if curve.empty:
        print(tag, '- log chưa có điểm đánh giá nào.')
        continue
    print('=' * 78)
    print(tag, '-', len(curve), 'lần đánh giá')
    print(curve.groupby('epoch')[['AUC1', 'AUC2', 'avgMAP']].max().round(2).to_string())
    best = curve.loc[curve['AUC1'].idxmax()]
    print(f'  Tốt nhất: epoch {int(best.epoch)}, step {int(best.step)} -> '
          f'AUC1={best.AUC1:.2f} AP1={best.AP1:.2f} AUC2={best.AUC2:.2f} avgMAP={best.avgMAP:.2f}')
    curve.to_csv(DRIVE_RESULT / f'curve_{tag}.csv', index=False)

    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(curve.index, curve['AUC1'], label='AUC1 (phân loại)')
        ax.plot(curve.index, curve['AUC2'], label='AUC2 (đối chiếu)')
        ax.axhline(PAPER_METRICS['AUC1'], ls='--', c='gray', lw=1, label='paper 88.02 (tham chiếu)')
        ax.set_xlabel('lần đánh giá'); ax.set_ylabel('AUC (%)')
        ax.set_title(tag); ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout(); fig.savefig(DRIVE_RESULT / f'curve_{tag}.png', dpi=120); plt.show()
    except Exception as exc:
        print('  Bỏ qua phần vẽ:', exc)

## 8. Copy Trọng Số Về Drive

Checkpoint đang ở `/content`, mất khi session kết thúc.

`epoch_checkpoints/` nặng vài trăm MB mỗi file × 10 epoch — chỉ copy khi cần phân tích theo epoch.

In [ ]:
COPY_EPOCH_CHECKPOINTS = False

for tag in (trained_tags or [f'baseline_seed{s}' for s in SEEDS]):
    src, dst = OUT_ROOT / tag, DRIVE_RESULT / tag
    if not src.exists():
        print('Bỏ qua', tag, '- chưa train.')
        continue
    dst.mkdir(parents=True, exist_ok=True)
    for name in [f'model_{tag}.pth', 'checkpoint.pth']:
        if (src / name).exists():
            print(f'{tag}: copying {name} ({(src / name).stat().st_size / 1e6:.0f} MB) ...', flush=True)
            shutil.copy2(src / name, dst / name)
    if COPY_EPOCH_CHECKPOINTS and (src / 'epoch_checkpoints').exists():
        (dst / 'epoch_checkpoints').mkdir(exist_ok=True)
        for f in sorted((src / 'epoch_checkpoints').glob('*.pth')):
            print('  ', f.name, flush=True)
            shutil.copy2(f, dst / 'epoch_checkpoints' / f.name)
    print('->', dst)

## 9. Bảng Tổng Hợp

Gom mọi `score_*.log` thành một bảng, cộng thêm dòng thống kê cho các lần baseline.

Cách đọc, và cũng là cách nên viết vào báo cáo:

- **`paper`** là dòng tham chiếu, chứng tỏ bản cài đặt trung thực. **Không** dùng làm cột hơn thua.
- **Các dòng `baseline_*`** là mốc so sánh thật. Độ lệch chuẩn của chúng là mức nhiễu.
- **`v0`** chỉ hơn/kém có ý nghĩa khi chênh lệch so với `baseline_ctrl` **lớn hơn** mức nhiễu đó.

In [ ]:
rows = []
for log in sorted(LOG_DIR.glob('score_*.log')):
    name = log.stem.replace('score_', '')
    scores = read_score(name)
    if scores:
        rows.append({'model': name, **scores})

if not rows:
    print('Chưa có kết quả nào. Chạy §5 trước.')
else:
    table = pd.DataFrame(rows).set_index('model').round(2)
    display(table)

    base = table[table.index.str.startswith('baseline')]
    if len(base) >= 2:
        print()
        print(f'Thống kê trên {len(base)} lần baseline ({", ".join(base.index)}):')
        for col in ['AUC1', 'avgMAP']:
            print(f'  {col:7s} trung bình {base[col].mean():6.2f}  '
                  f'độ lệch chuẩn {base[col].std():5.2f}  '
                  f'khoảng [{base[col].min():.2f}, {base[col].max():.2f}]')
        print()
        if 'v0' in table.index and 'baseline_ctrl' in table.index:
            print('v0 so với baseline_ctrl (cặp cùng script, cùng seed, khác đúng lambda):')
            for col in ['AUC1', 'AP1', 'AUC2', 'AP2', 'avgMAP']:
                delta = table.loc['v0', col] - table.loc['baseline_ctrl', col]
                noise = base[col].std() if col in base else float('nan')
                verdict = 'trong nhiễu' if abs(delta) < noise else 'VƯỢT nhiễu'
                print(f'  {col:7s} {delta:+6.2f}   (độ lệch chuẩn baseline {noise:.2f})  -> {verdict}')
    elif len(base) == 1:
        print()
        print('Mới có 1 lần baseline. Cần ít nhất 3 để tính độ lệch chuẩn — thêm seed vào SEEDS ở §1.')

    table.to_csv(DRIVE_RESULT / 'baseline_summary.csv')
    print()
    print('Saved:', DRIVE_RESULT / 'baseline_summary.csv')

## Ghi Chú

**Con số nào so với con số nào.** `v0` chỉ so được với `baseline_ctrl` — hai lần chạy đó cùng
script, cùng seed, cùng thứ tự dữ liệu, khác đúng một biến là λ. Các lần `baseline_seed*` phục vụ
việc khác: đo xem con số nhảy bao nhiêu khi **không đổi gì cả**.

**Đừng chọn lần chạy đẹp nhất.** Nếu chạy lại tới khi ra số ưng ý rồi lấy nó làm baseline, đó là
chọn kết quả theo tập test ở cấp độ lần-chạy. Dấu hiệu nhận biết: bạn đang nghĩ "chạy thêm lần nữa
xem có đẹp hơn không".

**Kết luận "chưa chứng minh được" vẫn là kết luận hợp lệ** và đáng viết vào báo cáo. Nghiên cứu
trung thực không bắt buộc phải tìm ra cải thiện.

**Chi tiết code:** `VadCLIP/baseline/README.md` liệt kê đủ 4 file đã sửa so với upstream và lý do
từng chỗ.